In [ ]:
 !pip install curl_cffi playwright pandas tqdm
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 21.4 MB/s eta 0:00:00
170.4 MiB [] 0% 0.0s170.4 MiB [] 0% 57.9s170.4 MiB [] 0% 34.1s170.4 MiB [] 0% 20.9s170.4 MiB [] 0% 15.8s170.4 MiB [] 0% 10.0s170.4 MiB [] 1% 6.8s170.4 MiB [] 2% 5.3s170.4 MiB [] 2% 4.6s170.4 MiB [] 3% 4.0s170.4 MiB [] 3% 4.1s170.4 MiB [] 4% 3.9s170.4 MiB [] 5% 3.2s170.4 MiB [] 6% 3.1s170.4 MiB [] 7% 2.9s170.4 MiB [] 8% 2.6s170.4 MiB [] 9% 2.6s170.4 MiB [] 10% 2.4s170.4 MiB [] 12% 2.2s170.4 MiB [] 13% 2.1s170.4 MiB [] 14% 2.0s170.4 MiB [] 15% 2.1s170.4 MiB [] 16% 2.0s170.4 MiB [] 17% 1.9s170.4 MiB [] 18% 1.8s170.4 MiB [] 19% 1.8s170.4 MiB [] 20% 1.8s170.4 MiB [] 21% 1.7s170.4 MiB [] 22% 1.6s170.4 MiB [] 24% 1.6s170.4 MiB [] 25% 1.5s170.4 MiB [] 27% 1.4s170.4 MiB [] 28% 1.3s170.4 MiB [] 30% 1.3s170.4 MiB [] 32% 1.2s170.4 MiB [] 33% 1.2s170.4 MiB [] 35% 1.1s170.4 MiB [] 37% 1.1s170.4 MiB [] 38% 1.0s170.4 MiB [] 40% 1.0s170.4 MiB [] 41% 0.9s170.4 MiB [] 43% 0.9s170.4 MiB [] 44% 0.9s170.4 MiB [] 45% 0.9s170.4 MiB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ganti baris ini di script:
OUTPUT_CSV = "/content/drive/MyDrive/StockTwits_CL_F_Historical.csv"

Mounted at /content/drive


In [ ]:
"""
============================================================
  StockTwits Historical Scraper — $CL_F (Crude Oil Futures)
  For use in: Google Colab / Local Python 3.8+
  Author  : Expert Data Engineer scaffold for thesis research
  Purpose : Collect Date, Body, Sentiment for FinBERT pipeline
============================================================

INSTALLATION (run once in a cell / terminal):
    pip install curl_cffi playwright pandas tqdm
    playwright install chromium

HOW IT WORKS
------------
StockTwits protects its REST API with Cloudflare and rate-limits
the public endpoint to 30 messages. This script uses a two-layer
strategy:

  Layer 1 – curl_cffi (TLS fingerprint spoofing)
    curl_cffi impersonates a real Chrome TLS handshake, which is
    enough to bypass Cloudflare's passive JS-challenge on the
    hidden paginated JSON endpoint:
        /api/2/streams/symbol/CL_F.json?max_id=<id>
    Each page returns up to 30 messages + the lowest message_id
    so the next page can be requested (cursor-based pagination).

  Layer 2 – Playwright headless browser (fallback)
    If Layer 1 returns 403 / 429 repeatedly (aggressive Cloudflare
    mode), the script falls back to a Playwright Chromium instance
    that renders the actual Next.js page, intercepts the XHR
    responses, and extracts the JSON payloads directly — exactly
    as a real user's browser would.

DATE FILTERING
--------------
StockTwits only lets you paginate *backwards* in time (newest →
oldest). The script starts from the current page, walks back via
max_id, and stops as soon as a message's created_at falls before
START_DATE. Messages outside [START_DATE, END_DATE] are dropped.

OUTPUT
------
StockTwits_CL_F_Historical.csv  with columns:
    message_id | created_at | body | sentiment
============================================================
"""

# ── Standard library ──────────────────────────────────────
import csv
import json
import random
import time
import logging
from datetime import datetime, timezone
from pathlib import Path

# ── Third-party ───────────────────────────────────────────
import pandas as pd
from tqdm import tqdm

# ── Optional imports (graceful degradation) ───────────────
try:
    from curl_cffi import requests as cffi_requests
    CURL_CFFI_AVAILABLE = True
except ImportError:
    CURL_CFFI_AVAILABLE = False
    import requests  # stdlib fallback (will likely 403)

try:
    from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeout
    PLAYWRIGHT_AVAILABLE = True
except ImportError:
    PLAYWRIGHT_AVAILABLE = False

# ══════════════════════════════════════════════════════════
#  USER CONFIGURATION  ← edit these values
# ══════════════════════════════════════════════════════════
TICKER        = "CL_F"
START_DATE    = datetime(2019, 1, 1,  tzinfo=timezone.utc)
END_DATE      = datetime(2026, 1, 1,  tzinfo=timezone.utc)
OUTPUT_CSV    = "StockTwits_CL_F_Historical.csv"

# Politeness settings (seconds)
MIN_DELAY     = 1.5   # minimum sleep between API calls
MAX_DELAY     = 3.5   # maximum sleep between API calls
MAX_RETRIES   = 5     # retries on 429 / 5xx before giving up
BACKOFF_BASE  = 8     # seconds for first back-off (doubles each retry)

# Playwright settings (Layer 2 fallback)
PW_HEADLESS   = True  # set False to watch the browser (useful for debugging)
PW_SLOW_MO    = 800   # milliseconds between Playwright actions

# ══════════════════════════════════════════════════════════
#  LOGGING
# ══════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ══════════════════════════════════════════════════════════
#  CONSTANTS
# ══════════════════════════════════════════════════════════
BASE_URL = "https://api.stocktwits.com/api/2/streams/symbol/{ticker}.json"

# Rotate real Chrome UA strings to reduce fingerprint repetition
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]

HEADERS_TEMPLATE = {
    "Accept"         : "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer"        : f"https://stocktwits.com/symbol/{TICKER}",
    "Origin"         : "https://stocktwits.com",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest" : "empty",
    "sec-fetch-mode" : "cors",
    "sec-fetch-site" : "same-origin",
}

# ══════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════

def _parse_dt(iso_string: str) -> datetime:
    """Parse ISO-8601 string from StockTwits into an aware datetime."""
    iso_string = iso_string.replace("Z", "+00:00")
    return datetime.fromisoformat(iso_string)


def _polite_sleep():
    """Random sleep to stay polite and avoid rate-limits."""
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def _build_url(ticker: str, max_id: int | None = None) -> str:
    url = BASE_URL.format(ticker=ticker)
    params = ["limit=30"]
    if max_id:
        params.append(f"max={max_id}")
    return url + "?" + "&".join(params)


def _extract_messages(data: dict) -> list[dict]:
    """Pull the fields we care about from a single API response dict."""
    results = []
    for msg in data.get("messages", []):
        sentiment_raw = msg.get("entities", {}).get("sentiment")
        if isinstance(sentiment_raw, dict):
            sentiment = sentiment_raw.get("basic", "").capitalize() or None
        else:
            sentiment = None

        results.append({
            "message_id": msg["id"],
            "created_at": msg["created_at"],
            "body"       : msg["body"],
            "sentiment"  : sentiment,
        })
    return results


# ══════════════════════════════════════════════════════════
#  LAYER 1 — curl_cffi  (TLS fingerprint spoofing)
# ══════════════════════════════════════════════════════════

class CurlCffiScraper:
    """
    Uses curl_cffi to impersonate Chrome's TLS fingerprint.
    This bypasses Cloudflare's passive bot checks that reject
    Python's default TLS stack.
    """

    def __init__(self, ticker: str):
        self.ticker  = ticker
        self.session = cffi_requests.Session(impersonate="chrome120")

    def fetch_page(self, max_id: int | None = None) -> dict | None:
        url     = _build_url(self.ticker, max_id)
        headers = {**HEADERS_TEMPLATE, "User-Agent": random.choice(USER_AGENTS)}

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = self.session.get(url, headers=headers, timeout=30)

                if resp.status_code == 200:
                    return resp.json()

                if resp.status_code == 429:
                    wait = BACKOFF_BASE * (2 ** (attempt - 1))
                    log.warning("Rate-limited (429). Waiting %ds …", wait)
                    time.sleep(wait)
                    continue

                if resp.status_code == 403:
                    log.error(
                        "403 Forbidden on attempt %d/%d. "
                        "Cloudflare active challenge detected.",
                        attempt, MAX_RETRIES,
                    )
                    time.sleep(BACKOFF_BASE * attempt)
                    continue

                log.warning("HTTP %d — skipping page.", resp.status_code)
                return None

            except Exception as exc:
                log.warning("Request error (attempt %d): %s", attempt, exc)
                time.sleep(BACKOFF_BASE * attempt)

        return None  # exhausted retries

    def scrape(
        self,
        start_date: datetime,
        end_date:   datetime,
        progress_bar: tqdm,
    ) -> list[dict]:
        all_messages: list[dict] = []
        max_id: int | None = None

        while True:
            data = self.fetch_page(max_id)

            if data is None:
                log.error("Layer 1 failed to fetch a page. Returning what we have.")
                break

            messages = _extract_messages(data)
            if not messages:
                log.info("No more messages returned. Pagination complete.")
                break

            # Pagination cursor — StockTwits returns the *min* id in this batch
            cursor = data.get("cursor", {})
            next_max = cursor.get("max") if cursor else None

            stop_scraping = False
            for msg in messages:
                msg_dt = _parse_dt(msg["created_at"])

                # Too new — skip but keep paginating
                if msg_dt > end_date:
                    continue

                # Too old — we've gone past our window, stop
                if msg_dt < start_date:
                    stop_scraping = True
                    break

                all_messages.append(msg)
                progress_bar.update(1)

            if stop_scraping:
                log.info("Reached START_DATE boundary. Stopping.")
                break

            if next_max is None or next_max == max_id:
                log.info("No more pages available from API.")
                break

            max_id = next_max
            _polite_sleep()

        return all_messages


# ══════════════════════════════════════════════════════════
#  LAYER 2 — Playwright  (headless browser fallback)
# ══════════════════════════════════════════════════════════

class PlaywrightScraper:
    """
    Full headless-browser fallback. Playwright launches a real
    Chromium instance, navigates to the StockTwits symbol page,
    and intercepts XHR/fetch responses containing the JSON
    message payloads — exactly like a real user's browser.

    Because this renders a real browser, it defeats virtually all
    Cloudflare JS-challenge variants (IUAM, managed challenge, etc.)
    at the cost of being slower (~3-5× slower than Layer 1).
    """

    def __init__(self, ticker: str):
        self.ticker = ticker
        self.url    = f"https://stocktwits.com/symbol/{ticker}"

    def scrape(
        self,
        start_date: datetime,
        end_date:   datetime,
        progress_bar: tqdm,
    ) -> list[dict]:

        if not PLAYWRIGHT_AVAILABLE:
            log.error("Playwright not installed. Run: pip install playwright && playwright install chromium")
            return []

        all_messages : list[dict] = []
        intercepted  : list[dict] = []
        reached_start             = False

        def handle_response(response):
            """Intercept API calls the page makes and grab the JSON."""
            if f"/streams/symbol/{self.ticker}" in response.url:
                try:
                    body = response.json()
                    if "messages" in body:
                        intercepted.append(body)
                except Exception:
                    pass

        with sync_playwright() as pw:
            browser = pw.chromium.launch(headless=PW_HEADLESS, slow_mo=PW_SLOW_MO)
            context = browser.new_context(
                user_agent=random.choice(USER_AGENTS),
                viewport={"width": 1280, "height": 900},
                locale="en-US",
                timezone_id="America/New_York",
            )
            page = context.new_page()
            page.on("response", handle_response)

            log.info("[PW] Navigating to %s", self.url)
            page.goto(self.url, wait_until="networkidle", timeout=60_000)

            # StockTwits lazy-loads via infinite scroll — we keep
            # scrolling to the bottom until we have enough data or
            # reach our date boundary.
            prev_count = 0
            no_new_count = 0

            while not reached_start:
                # Process any newly intercepted payloads
                while intercepted:
                    payload  = intercepted.pop(0)
                    messages = _extract_messages(payload)

                    for msg in messages:
                        msg_dt = _parse_dt(msg["created_at"])

                        if msg_dt > end_date:
                            continue
                        if msg_dt < start_date:
                            reached_start = True
                            break

                        all_messages.append(msg)
                        progress_bar.update(1)

                    if reached_start:
                        break

                if reached_start:
                    break

                # Scroll down to trigger next batch
                page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                time.sleep(random.uniform(2.0, 3.5))

                # Detect stall (no new messages for 3 consecutive scrolls)
                if len(all_messages) == prev_count:
                    no_new_count += 1
                    if no_new_count >= 3:
                        log.info("[PW] No new messages after %d scrolls. Done.", no_new_count)
                        break
                else:
                    no_new_count = 0

                prev_count = len(all_messages)

            browser.close()

        return all_messages


# ══════════════════════════════════════════════════════════
#  CHECKPOINT  (resume-safe incremental save)
# ══════════════════════════════════════════════════════════

def save_checkpoint(messages: list[dict], path: str):
    """Append new messages to the CSV so progress survives crashes."""
    file_path = Path(path)
    write_header = not file_path.exists()

    with open(file_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["message_id","created_at","body","sentiment"])
        if write_header:
            writer.writeheader()
        writer.writerows(messages)


def load_seen_ids(path: str) -> set[int]:
    """Load already-scraped message IDs to avoid duplicates on resume."""
    p = Path(path)
    if not p.exists():
        return set()
    df = pd.read_csv(p, usecols=["message_id"])
    return set(df["message_id"].tolist())


# ══════════════════════════════════════════════════════════
#  MAIN ORCHESTRATOR
# ══════════════════════════════════════════════════════════

def main():
    log.info("━━━ StockTwits $%s Scraper ━━━", TICKER)
    log.info("Date window : %s  →  %s", START_DATE.date(), END_DATE.date())
    log.info("Output file : %s", OUTPUT_CSV)

    if not CURL_CFFI_AVAILABLE and not PLAYWRIGHT_AVAILABLE:
        log.error(
            "Neither curl_cffi nor playwright is installed.\n"
            "Install with:\n"
            "    pip install curl_cffi playwright pandas tqdm\n"
            "    playwright install chromium"
        )
        return

    seen_ids = load_seen_ids(OUTPUT_CSV)
    log.info("Resuming — %d messages already in CSV.", len(seen_ids))

    collected: list[dict] = []

    with tqdm(unit=" msgs", desc="Scraping", dynamic_ncols=True) as pbar:

        # ── Layer 1 attempt ───────────────────────────────
        if CURL_CFFI_AVAILABLE:
            log.info("Trying Layer 1: curl_cffi TLS-fingerprint bypass …")
            scraper = CurlCffiScraper(TICKER)
            collected = scraper.scrape(START_DATE, END_DATE, pbar)

        # ── Layer 2 fallback if Layer 1 returned nothing ──
        if not collected:
            log.warning("Layer 1 yielded 0 messages. Falling back to Layer 2: Playwright …")
            if PLAYWRIGHT_AVAILABLE:
                scraper = PlaywrightScraper(TICKER)
                collected = scraper.scrape(START_DATE, END_DATE, pbar)
            else:
                log.error("Playwright not available. Cannot fall back. Exiting.")
                return

    # ── De-duplicate against existing CSV ─────────────────
    new_messages = [m for m in collected if m["message_id"] not in seen_ids]
    log.info("New messages this run : %d  (skipped %d duplicates)",
             len(new_messages), len(collected) - len(new_messages))

    if new_messages:
        save_checkpoint(new_messages, OUTPUT_CSV)

    # ── Final tidy-up: sort by date, deduplicate the CSV ──
    final_df = pd.read_csv(OUTPUT_CSV)
    before   = len(final_df)
    final_df.drop_duplicates(subset=["message_id"], inplace=True)
    final_df.sort_values("created_at", ascending=True, inplace=True)
    final_df.to_csv(OUTPUT_CSV, index=False)

    log.info("━━━ DONE ━━━")
    log.info("Total messages in CSV : %d  (removed %d dupes during cleanup)",
             len(final_df), before - len(final_df))
    log.info("Saved to → %s", Path(OUTPUT_CSV).resolve())

    # Quick stats
    has_sentiment = final_df["sentiment"].notna().sum()
    log.info("Messages with sentiment label : %d / %d (%.1f%%)",
             has_sentiment, len(final_df), 100 * has_sentiment / max(len(final_df), 1))


# ══════════════════════════════════════════════════════════
if __name__ == "__main__":
    main()

Scraping: 41568 msgs [1:35:42,  7.24 msgs/s]
